# **Modelo LightFM - Features categóricas**
### Proyecto Hito 2
### Sistemas Recomendadores IIC3633-1 2025-2
### **Grupo 3:** 

- Nicolás Antonio Bueno Abett de la Torre 

- Felipe Andrés Fuentes González

- Jorge Andrés Jacque Palma

- Francisco Nicolás Solís Gormaz

In [4]:
# !pip install git+https://github.com/daviddavo/lightfm

In [5]:
import os
import sys
import itertools
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lightfm
from lightfm import LightFM
from lightfm.data import Dataset
from sklearn.model_selection import train_test_split
from lightfm import cross_validation
from lightfm.evaluation import precision_at_k as lightfm_prec_at_k
from lightfm.evaluation import recall_at_k as lightfm_recall_at_k

print("System version: {}".format(sys.version))
print("LightFM version: {}".format(lightfm.__version__))


System version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
LightFM version: 1.17


La idea ahora será alimentar LightFM con features de los videojuegos. Seguimos la misma estructura del notebook base

In [6]:
DATA_PATH_RATINGS = "video_game_reviews_with_userid_clean.csv"
DATA_PATH_METADATA = "video_game_reviews.csv"

SEED = 42
TEST_PERCENTAGE = 0.25
K = 10
NO_THREADS = 8
NO_EPOCHS = 20

ratings = pd.read_csv(DATA_PATH_RATINGS)
metadata = pd.read_csv(DATA_PATH_METADATA)

ratings = ratings.rename(columns={
    "user_id": "userID",
    "item_id": "itemID",
    "rating": "rating"
})

ratings["userID"] = ratings["userID"].astype(str)
ratings["itemID"] = ratings["itemID"].astype(str)
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")

ratings = ratings.drop_duplicates(subset=["userID", "itemID"])
print(f"Ratings: {ratings.shape}")


Ratings: (39441, 3)


Ahora, la idea va a ser rescatar los nombres de los juegos, generar un mapeo y así alimentar features a LightFM

In [7]:
# metadata con nombres de juegos
metadata = metadata.rename(columns={"Game Title": "GameTitle"})
metadata["GameTitle"] = metadata["GameTitle"].astype(str).str.strip()
print(f"Metadata: {metadata.shape}")

feature_cols = [
    "Genre", "Platform", "Developer", "Publisher",
    "Game Mode", "Multiplayer", "Requires Special Device",
    "Age Group Targeted"
]

def make_features(row):
    feats = []
    for c in feature_cols:
        if c in row and pd.notna(row[c]):
            feats.append(f"{c}={str(row[c]).strip()}")
    return feats

metadata["features"] = metadata.apply(make_features, axis=1)

Metadata: (47774, 18)


Ahora, el mapeo entre idem_id y el título.

In [8]:
unique_items = sorted(ratings["itemID"].unique())
metadata = metadata.head(len(unique_items)).copy()
metadata["itemID"] = unique_items[:len(metadata)]
metadata["itemID"] = metadata["itemID"].astype(str)

Ya con esto podemos hacer el split, del mismo modo que hicimos el del código base.

In [9]:
def stratified_user_split(df, test_size=0.25, seed=42):
    train_parts, test_parts = [], []
    for user, grp in df.groupby("userID"):
        if len(grp) < 2:
            train_parts.append(grp)
            continue
        tr, te = train_test_split(grp, test_size=test_size, random_state=seed)
        train_parts.append(tr)
        test_parts.append(te)
    return pd.concat(train_parts), pd.concat(test_parts)

train_df, test_df = stratified_user_split(ratings, test_size=TEST_PERCENTAGE, seed=SEED)
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (28451, 3), Test: (10990, 3)


Ahora ya podemos construir el dataset con item_features. También, generamos las interacciones.

In [10]:
all_users = ratings["userID"].unique()
all_items = ratings["itemID"].unique()
all_features = sorted({f for feats in metadata["features"].dropna() for f in feats})

dataset2 = Dataset()
dataset2.fit(users=all_users, items=all_items, item_features=all_features)

# interacciones
(train_interactions2, _) = dataset2.build_interactions(train_df[["userID", "itemID", "rating"]].values)
(test_interactions2, _)  = dataset2.build_interactions(test_df[["userID", "itemID", "rating"]].values)


In [11]:

def item_features_gen():
    for row in metadata.itertuples(index=False):
        feats = getattr(row, "features")
        if isinstance(feats, list):
            yield (getattr(row, "itemID"), feats)
        else:
            yield (getattr(row, "itemID"), [])

item_features = dataset2.build_item_features(item_features_gen())
print(f"item features shape: {item_features.shape}")

item features shape: (40, 84)


Ahora podemos hacer el modelo con los features.

In [12]:
model2 = LightFM(
    loss='warp',
    no_components=16,
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=np.random.RandomState(SEED)
)

model2.fit(
    train_interactions2,
    item_features=item_features,
    epochs=NO_EPOCHS,
    num_threads=NO_THREADS
)

prec2 = lightfm_prec_at_k(
    model2, test_interactions2,
    train_interactions=train_interactions2,
    item_features=item_features,
    k=K, num_threads=NO_THREADS
).mean()

rec2 = lightfm_recall_at_k(
    model2, test_interactions2,
    train_interactions=train_interactions2,
    item_features=item_features,
    k=K, num_threads=NO_THREADS
).mean()

print(f"Precision@{K}: {prec2:.4f}")
print(f"Recall@{K}:    {rec2:.4f}")


Precision@10: 0.1218
Recall@10:    0.3285


In [13]:
from tqdm import tqdm

def evaluate_lightfm_models(
    train_interactions,
    test_interactions,
    param_grid,
    K=10,
    num_threads=8,
    max_epochs=20,
    random_state=42
):
    results = []

    for params in tqdm(param_grid, desc="evaluando hiperparámetros"):
        # modelo
        model = LightFM(
            loss=params['loss'],
            no_components=params['no_components'],
            learning_rate=params['learning_rate'],
            random_state=np.random.RandomState(random_state)
        )

        # entrenamiento
        model.fit(train_interactions, epochs=max_epochs, num_threads=num_threads)

        # evaluación
        prec = lightfm_prec_at_k(
            model, test_interactions,
            train_interactions=train_interactions,
            k=K, num_threads=num_threads
        ).mean()

        rec = lightfm_recall_at_k(
            model, test_interactions,
            train_interactions=train_interactions,
            k=K, num_threads=num_threads
        ).mean()

        results.append({
            'params': params,
            f'precision@{K}': prec,
            f'recall@{K}': rec
        })

    return pd.DataFrame(results)


In [14]:
param_grid = [
    {'loss': 'warp', 'no_components': 16, 'learning_rate': 0.05},
    {'loss': 'warp', 'no_components': 32, 'learning_rate': 0.05},
    {'loss': 'warp', 'no_components': 32, 'learning_rate': 0.1},
    {'loss': 'bpr',  'no_components': 32, 'learning_rate': 0.05},
    {'loss': 'bpr',  'no_components': 64, 'learning_rate': 0.05},
]

results = evaluate_lightfm_models(
    train_interactions=train_interactions2,
    test_interactions=test_interactions2,
    param_grid=param_grid,
    K=10,
    num_threads=8,
    max_epochs=20,
    random_state=SEED
)

print(results.sort_values(by='precision@10', ascending=False))


evaluando hiperparámetros: 100%|██████████| 5/5 [00:56<00:00, 11.34s/it]

                                              params  precision@10  recall@10
1  {'loss': 'warp', 'no_components': 32, 'learnin...      0.122500   0.328211
3  {'loss': 'bpr', 'no_components': 32, 'learning...      0.122500   0.330456
2  {'loss': 'warp', 'no_components': 32, 'learnin...      0.122433   0.329156
4  {'loss': 'bpr', 'no_components': 64, 'learning...      0.120867   0.324911
0  {'loss': 'warp', 'no_components': 16, 'learnin...      0.120767   0.325706


In [20]:
LOSS_FUNCTION = 'warp'
NO_COMPONENTS = 32
LEARNING_RATE = 0.05
ITEM_ALPHA = 1e-6
USER_ALPHA = 1e-6


In [21]:
model_final = LightFM(
    loss=LOSS_FUNCTION,
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    item_alpha=ITEM_ALPHA,
    user_alpha=USER_ALPHA,
    random_state=SEED
)

model_final.fit(train_interactions2, epochs=20, num_threads=8)

Ya tenemos el modelo final.

In [22]:
eval_precision_lfm = lightfm_prec_at_k(
    model_final,
    test_interactions2,
    train_interactions=train_interactions2,
    k=K
).mean()

eval_recall_lfm = lightfm_recall_at_k(
    model_final,
    test_interactions2,
    train_interactions=train_interactions2,
    k=K
).mean()

print(f"Precision@{K}: {eval_precision_lfm:.4f}")
print(f"Recall@{K}:    {eval_recall_lfm:.4f}")



Precision@10: 0.1215
Recall@10:    0.3258


In [ ]:
user_mapping, _, item_mapping, _ = dataset2.mapping()
print(list(user_mapping.keys())[:5])


['861', '1295', '1131', '1096', '1639']


In [ ]:
# mapeos para convertir entre índices internos y ids originales
user_id_map, user_feat_map, item_id_map, item_feat_map = dataset2.mapping()
inv_item_id_map = {v: k for k, v in item_id_map.items()}
inv_user_id_map = {v: k for k, v in user_id_map.items()}

# recomendar para un usuario (por indice interno de LightFM)
def recommend_for_user(model, user_internal_id, train_interactions, K=10,
                       user_features=None, item_features=None):
    n_users, n_items = train_interactions.shape

    # predice para tds los items
    item_ids = np.arange(n_items)
    scores = model.predict(
        user_ids=user_internal_id,
        item_ids=item_ids,
        user_features=user_features,
        item_features=item_features
    )

    # "enmascara" items ya vistos por ese usuario en el set de training
    known_items = train_interactions.tocsr()[user_internal_id].indices
    scores[known_items] = -np.inf
    valid_mask = np.isfinite(scores)
    num_candidates = int(valid_mask.sum())

    # Top-K, 10 es lo que usamos pero igual es mejor dejarlo parametrizado por si acaso
    K_eff = min(K, num_candidates) if num_candidates > 0 else 0
    top_items = np.argsort(-scores)[:K_eff]

    # convierte indices internos a ids originales
    rec_item_ids = [inv_item_id_map[i] for i in top_items]

    return {
        "requested_K": K,
        "available_candidates": num_candidates,
        "returned_K": K_eff,
        "internal_item_indices": top_items.tolist(),
        "item_ids": rec_item_ids
    }

# como ejemplo, la idea es elegir un usuario válido, por índice interno
example_user_internal_id = 2

res = recommend_for_user(
    model=model_final,
    user_internal_id=example_user_internal_id,
    train_interactions=train_interactions2,
    K=K
)

print(f"usuario interno: {example_user_internal_id} (ID original: {inv_user_id_map.get(example_user_internal_id)})")
print(f"candidatos disponibles (no vistos): {res['available_candidates']}")
print(f"solicitados K={res['requested_K']}, Devueltos: {res['returned_K']}")
print("recomendaciones (itemID):")
for iid in res["item_ids"]:
    print("  -", iid)


usuario interno: 2 (ID original: 1131)
candidatos disponibles (no vistos): 31
solicitados K=10, Devueltos: 10
recomendaciones (itemID):
  - 27
  - 12
  - 39
  - 21
  - 26
  - 28
  - 18
  - 2
  - 10
  - 17


In [26]:
rec_df = pd.DataFrame({"itemID": res["item_ids"]})
rec_df = rec_df.merge(metadata[["itemID", "GameTitle"]], on="itemID", how="left")

print("recomendaciones para el usuario 1131:")
display(rec_df)


recomendaciones para el usuario 1131:


,itemID,GameTitle
0,27,Stardew Valley
1,12,Bioshock Infinite
2,39,Fortnite
3,21,Fall Guys
4,26,Fall Guys
5,28,Spelunky 2
6,18,1000-Piece Puzzle
7,2,Street Fighter V
8,10,The Sims 4
9,17,Sid Meier’s Civilization VI


## Métricas

Como se dijo en el notebook base, usamos las mismas de siempre.

In [27]:
def precision_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rec_k)

def recall_at_k(rec_k, rel_set):
    if len(rel_set) == 0:
        return 0.0
    hits = sum((i in rel_set) for i in rec_k)
    return hits / len(rel_set)

def ndcg_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    dcg = 0.0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            dcg += 1.0 / np.log2(rank + 1)
    ideal = min(len(rel_set), len(rec_k))
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal + 1))
    return (dcg / idcg) if idcg > 0 else 0.0

def hit_score_at_k(rec_k, rel_set):
    return 1.0 if any((i in rel_set) for i in rec_k) else 0.0

def map_at_k(rec_k, rel_set):
    if len(rec_k) == 0:
        return 0.0
    ap_sum = 0.0
    hits = 0
    for rank, it in enumerate(rec_k, start=1):
        if it in rel_set:
            hits += 1
            ap_sum += hits / rank
    return ap_sum / len(rel_set) if len(rel_set) > 0 else 0.0

def diversity_at_k(rec_k_dict, info_videojuegos):
    diversidades = []
    for uid, recs in rec_k_dict.items():
        generos = {info_videojuegos[i][1] for i in recs if i in info_videojuegos}
        diversidades.append(len(generos))
    return np.mean(diversidades) if diversidades else np.nan

In [29]:
itemid2title = dict(metadata[["itemID", "GameTitle"]].drop_duplicates("itemID").values)
itemid2genre = dict(metadata[["itemID", "Genre"]].drop_duplicates("itemID").values)

info_videojuegos = {}
for internal_idx, original_item_id in inv_item_id_map.items():
    title = itemid2title.get(str(original_item_id))
    genre = itemid2genre.get(str(original_item_id))
    info_videojuegos[internal_idx] = (title, genre)

def evaluar_metricas(
    model,
    train_interactions,
    test_interactions,
    K=10,
    num_threads=NO_THREADS,
    info_videojuegos=None
):
    n_users, n_items = train_interactions.shape
    train_csr = train_interactions.tocsr()
    test_csr  = test_interactions.tocsr()

    rec_k_dict = {}
    rows_user = []

    for u in range(n_users):
        # relevantes en test para el usuario
        rel_set = set(test_csr[u].indices)
        item_ids = np.arange(n_items)

        scores = model.predict(
            user_ids=u,
            item_ids=item_ids,
            num_threads=num_threads
        )

        # enmascarar items ya vistos en train
        scores[train_csr[u].indices] = -np.inf

        valid = np.isfinite(scores)
        if not valid.any():
            rec_k = []
        else:
            K_eff = min(K, int(valid.sum()))
            top_idx = np.argsort(-scores)[:K_eff]
            rec_k = list(top_idx)

        rec_k_dict[u] = rec_k

        prec = precision_at_k(rec_k, rel_set)
        rec  = recall_at_k(rec_k, rel_set)
        ndcg = ndcg_at_k(rec_k, rel_set)
        hit  = hit_score_at_k(rec_k, rel_set)
        m_ap = map_at_k(rec_k, rel_set)

        rows_user.append({
            "user_internal": u,
            "relevantes_test": len(rel_set),
            "K_devueltos": len(rec_k),
            "precision@K": prec,
            "recall@K": rec,
            "ndcg@K": ndcg,
            "hit_score@K": hit,
            "map@K": m_ap
        })

    df_users = pd.DataFrame(rows_user)
    global_metrics = {
        "users_evaluated": int(df_users.shape[0]),
        "precision@K": float(df_users["precision@K"].mean()) if not df_users.empty else np.nan,
        "recall@K": float(df_users["recall@K"].mean()) if not df_users.empty else np.nan,
        "ndcg@K": float(df_users["ndcg@K"].mean()) if not df_users.empty else np.nan,
        "hit_score@K": float(df_users["hit_score@K"].mean()) if not df_users.empty else np.nan,
        "map@K": float(df_users["map@K"].mean()) if not df_users.empty else np.nan
    }

    if info_videojuegos is not None and len(info_videojuegos) > 0:
        global_metrics["diversity@K"] = float(diversity_at_k(rec_k_dict, info_videojuegos))
    else:
        global_metrics["diversity@K"] = np.nan

    df_global = pd.DataFrame([global_metrics])

    return df_global

df_global = evaluar_metricas(
    model=model_final,
    train_interactions=train_interactions2,
    test_interactions=test_interactions2,
    K=K,
    num_threads=NO_THREADS,
    info_videojuegos=info_videojuegos
)

display(df_global)

,users_evaluated,precision@K,recall@K,ndcg@K,hit_score@K,map@K,diversity@K
0,3000,0.121533,0.32585,0.225062,0.766333,0.117696,6.677333


Referencias adicionales
- https://www.stepbystepdatascience.com/hybrid-recommender-lightfm-python